# Day 2: SQL JOINs — Interactive Practice

## Objective
Learn how to combine data from multiple tables using SQL JOINs. By the end of this notebook, you will be comfortable writing `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, `FULL OUTER JOIN`, self-joins, and multi-table joins.

## Table of Contents
1. [Connection Setup](#connection-setup)
2. [INNER JOIN — Only Matching Rows](#inner-join--only-matching-rows)
3. [LEFT JOIN — Keep All Rows from the Left](#left-join--keep-all-rows-from-the-left)
4. [Filtering with Joins — Finding Orphan Records](#filtering-with-joins--finding-orphan-records)
5. [RIGHT JOIN — The Mirror Image](#right-join--the-mirror-image)
6. [FULL OUTER JOIN — Everything from Both Sides](#full-outer-join--everything-from-both-sides)
7. [Self-Join — The Manager Hierarchy](#self-join--the-manager-hierarchy)
8. [Multi-Table Join — employees → departments → sales → products](#multi-table-join--employees--departments--sales--products)
9. [Try It Yourself Exercises](#try-it-yourself-exercises)

## Connection Setup

We use the same `run_query` helper from Day 1. It connects to the PostgreSQL database, executes a query, returns results as a pandas DataFrame, and always closes the connection.

**Tables we will use today:**
- `company.employees` — emp_id, first_name, last_name, email, department_id, salary, hire_date, manager_id, is_active
- `company.departments` — dept_id, dept_name, location
- `company.sales` — sale_id, employee_id, product_id, quantity, sale_date
- `company.products` — product_id, product_name, unit_price, category

In [ ]:
import psycopg2
import pandas as pd

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query and return results as a DataFrame."""
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        df = pd.read_sql_query(sql, conn)
        return df
    finally:
        conn.close()

print("Helper function defined. Ready for JOINs!")

Let's quickly remind ourselves of the two tables we will join first. Always inspect your tables before writing a JOIN — it helps you predict the result.

In [ ]:
# Quick peek: employees (just the columns relevant for our join)
run_query("""
    SELECT emp_id, first_name, last_name, department_id 
    FROM company.employees 
    ORDER BY emp_id 
    LIMIT 10
""")

**Expected output:** 10 rows showing emp_id, first_name, last_name, and department_id. Notice that some employees have department_id = 1 (Engineering), some = 2 (HR), etc. Also check if any have `NULL`.

In [ ]:
# Quick peek: departments
run_query("""
    SELECT * FROM company.departments 
    ORDER BY dept_id
""")

**Expected output:** All departments with their dept_id, dept_name, and location. This is a small table (around 6 departments). Note which department IDs exist.

## INNER JOIN — Only Matching Rows

An `INNER JOIN` returns **only rows where the join key matches in both tables**. If an employee has no department, or a department_id that doesn't exist in the departments table, that employee will **not appear** in the result.

**Plain English:** "Show me employees alongside their department name — but only if they actually belong to a department."

**Syntax note:** `JOIN` without a qualifier means `INNER JOIN`. They are identical.

In [ ]:
# INNER JOIN: employees + departments
run_query("""
    SELECT 
        e.emp_id,
        e.first_name,
        e.last_name,
        e.department_id,
        d.dept_name,
        d.location
    FROM company.employees e
    INNER JOIN company.departments d 
        ON e.department_id = d.dept_id
    ORDER BY e.emp_id
""")

**Expected output:** Employees with valid departments only. Each row shows the employee's info alongside their department name and location. Employees with `NULL` department_id or invalid department_ids will be missing.

### How many rows did we lose?

Let's count the employees before and after the join to see exactly how many rows disappeared.

In [ ]:
# Count total employees (before join)
run_query("""
    SELECT COUNT(*) AS total_employees 
    FROM company.employees
""")

**Expected output:** A single number (around 30 employees total).

In [ ]:
# Count employees with matching departments (after INNER JOIN)
run_query("""
    SELECT COUNT(*) AS matched_employees
    FROM company.employees e
    INNER JOIN company.departments d 
        ON e.department_id = d.dept_id
""")

**Expected output:** A number smaller than the total. The difference tells you how many employees have no valid department.

**Key insight:** If `total_employees - matched_employees > 0`, those missing employees have either `NULL` as their `department_id` or a department_id that doesn't exist in the departments table. INNER JOIN silently dropped them!

In [ ]:
# Let's see who got dropped: employees without a valid department
run_query("""
    SELECT emp_id, first_name, last_name, department_id
    FROM company.employees
    WHERE department_id IS NULL 
       OR department_id NOT IN (SELECT dept_id FROM company.departments)
    ORDER BY emp_id
""")

**Expected output:** The employees that disappeared from the INNER JOIN. Likely Paula Morris (department_id = NULL) and possibly others with invalid department references. This is why INNER JOIN can be dangerous for reports — you might not realize you're missing people!

## LEFT JOIN — Keep All Rows from the Left

A `LEFT JOIN` returns **ALL rows from the left table** (the one after `FROM`), plus matching rows from the right table. Where no match exists, the right-side columns are filled with `NULL`.

**Plain English:** "Show me ALL employees, and their department name if they have one."

**Why LEFT JOIN is the default in real work:** You rarely want to silently lose data. If a new employee hasn't been assigned a department yet, you still want to see them in your report.

In [ ]:
# LEFT JOIN: ALL employees, with department info when available
run_query("""
    SELECT 
        e.emp_id,
        e.first_name,
        e.last_name,
        e.department_id,
        d.dept_name,
        d.location
    FROM company.employees e
    LEFT JOIN company.departments d 
        ON e.department_id = d.dept_id
    ORDER BY e.emp_id
""")

**Expected output:** ALL employees appear (same count as `total_employees` from earlier). Most will have dept_name and location filled in. The employees without a valid department will show `NULL` in the dept_name and location columns.

**Compare with the INNER JOIN result:** The INNER JOIN had fewer rows. The LEFT JOIN has everyone. The rows that were missing from INNER JOIN now appear with NULL values on the right side.

In [ ]:
# Verify: COUNT should match total_employees from earlier
run_query("""
    SELECT COUNT(*) AS all_employees_count
    FROM company.employees e
    LEFT JOIN company.departments d 
        ON e.department_id = d.dept_id
""")

**Expected output:** Same number as `total_employees` from the count earlier. This proves LEFT JOIN preserves all left-table rows.

**Understanding NULL in LEFT JOIN:** When there's no matching row on the right side, PostgreSQL fills those columns with NULL. This is not an error — it means "this employee doesn't have a department." You can use these NULLs to find problems in your data (we'll do that next).

## Filtering with Joins — Finding Orphan Records

One of the most practical uses of LEFT JOIN is finding **orphan records** — rows on the left side that have no match on the right. This is a powerful data quality check.

**Pattern:** LEFT JOIN + `WHERE right_table.column IS NULL`

**Plain English:** "Show me employees who don't belong to any department."

In [ ]:
# Find employees without a valid department
run_query("""
    SELECT 
        e.emp_id,
        e.first_name,
        e.last_name,
        e.department_id
    FROM company.employees e
    LEFT JOIN company.departments d 
        ON e.department_id = d.dept_id
    WHERE d.dept_id IS NULL
    ORDER BY e.emp_id
""")

**Expected output:** Only the employees with no matching department. This is a much smaller list than the full LEFT JOIN result. These are your "orphan" records.

**Why this works:** The LEFT JOIN first includes ALL employees. Then the WHERE clause filters to only those rows where `d.dept_id` ended up as NULL — meaning no match was found. This is the classic pattern for finding data quality issues.

**Real-world use cases:**
- Find orders with no customer
- Find products with no category
- Find students with no enrolled course
- Find sales transactions with no assigned salesperson

## RIGHT JOIN — The Mirror Image

A `RIGHT JOIN` is a LEFT JOIN with the tables swapped. It keeps ALL rows from the **right** table (the one after `JOIN`), and matches from the left.

**Practical tip:** RIGHT JOIN is almost never used in practice. Just swap the table order and use LEFT JOIN instead — it's more readable.

Let's prove they produce the same result.

In [ ]:
# RIGHT JOIN: ALL departments, with employees when they exist
run_query("""
    SELECT 
        e.emp_id,
        e.first_name,
        e.last_name,
        d.dept_id,
        d.dept_name
    FROM company.employees e
    RIGHT JOIN company.departments d 
        ON e.department_id = d.dept_id
    ORDER BY d.dept_id, e.emp_id
""")

**Expected output:** ALL departments appear. Most will have employees listed. If any department has no employees, those rows will show NULL for emp_id, first_name, and last_name.

In [ ]:
# Equivalent LEFT JOIN (tables swapped): Same result, more readable
run_query("""
    SELECT 
        e.emp_id,
        e.first_name,
        e.last_name,
        d.dept_id,
        d.dept_name
    FROM company.departments d
    LEFT JOIN company.employees e 
        ON e.department_id = d.dept_id
    ORDER BY d.dept_id, e.emp_id
""")

**Expected output:** Identical to the RIGHT JOIN above. The only difference is the column order (dept_id/dept_name come first now because `departments` is the left table).

**Takeaway:** `A RIGHT JOIN B` = `B LEFT JOIN A`. Just swap the tables and use LEFT JOIN. Most teams never use RIGHT JOIN in production code.

## FULL OUTER JOIN — Everything from Both Sides

A `FULL OUTER JOIN` returns **ALL rows from both tables**. Matched rows get combined. Unmatched rows from either side appear with NULLs on the other side.

**Plain English:** "Show me ALL employees (even without departments) AND all departments (even without employees)."

**Use case:** A comprehensive audit — finding every unmatched record on both sides at once.

In [ ]:
# FULL OUTER JOIN: every employee and every department
run_query("""
    SELECT 
        e.emp_id,
        e.first_name,
        e.last_name,
        e.department_id,
        d.dept_id,
        d.dept_name,
        d.location
    FROM company.employees e
    FULL OUTER JOIN company.departments d 
        ON e.department_id = d.dept_id
    ORDER BY 
        CASE WHEN e.emp_id IS NULL THEN 1 ELSE 0 END,
        e.emp_id,
        d.dept_id
""")

**Expected output:** 
- Most rows: employees with their departments (both sides filled)
- Some rows: employees with NULL dept_name (employee has no valid department)
- Possibly some rows: NULL emp_id with a dept_name (department has no employees)

**The ORDER BY trick:** `CASE WHEN e.emp_id IS NULL THEN 1 ELSE 0 END` pushes unmatched departments (where emp_id is NULL) to the bottom so you can easily see them.

**Key insight:** FULL OUTER JOIN = LEFT JOIN results + RIGHT JOIN results, combined and deduplicated. It's the most inclusive join type.

In [ ]:
# Use case: Find ALL unmatched records from both sides
run_query("""
    SELECT 
        e.emp_id,
        e.first_name,
        e.last_name,
        d.dept_name,
        d.location,
        CASE 
            WHEN e.emp_id IS NULL THEN 'Department has no employees'
            WHEN d.dept_id IS NULL THEN 'Employee has no department'
        END AS issue_type
    FROM company.employees e
    FULL OUTER JOIN company.departments d 
        ON e.department_id = d.dept_id
    WHERE e.emp_id IS NULL 
       OR d.dept_id IS NULL
    ORDER BY issue_type, e.emp_id
""")

**Expected output:** Only the problematic rows, with a clear `issue_type` label telling you whether it's an employee without a department or a department without employees. This is a handy data quality report.

## Self-Join — The Manager Hierarchy

A **self-join** joins a table to itself. This is useful for hierarchical data, like employees and their managers — where both are stored in the same `employees` table.

**How it works:** The `employees` table has a `manager_id` column that points to another employee's `emp_id`. We join `employees AS e` (the worker) to `employees AS m` (the manager).

**Why aliases are mandatory:** When the same table appears twice, you MUST use aliases so PostgreSQL knows which instance you mean. `e.first_name` is the employee's name, `m.first_name` is the manager's name.

In [ ]:
# Self-join: Show each employee alongside their manager's name
run_query("""
    SELECT 
        e.emp_id,
        e.first_name || ' ' || e.last_name AS employee_name,
        e.department_id,
        m.first_name || ' ' || m.last_name AS manager_name
    FROM company.employees e
    INNER JOIN company.employees m 
        ON e.manager_id = m.emp_id
    ORDER BY e.emp_id
    LIMIT 20
""")

**Expected output:** Most employees with their manager's full name. For example, Bob Martinez → Alice Chen (Bob's manager is Alice). Directors with `manager_id = NULL` will NOT appear because INNER JOIN requires a match.

**The `||` operator:** This is PostgreSQL string concatenation. `first_name || ' ' || last_name` joins the first name, a space, and the last name into one string.

In [ ]:
# LEFT JOIN version: Include directors who have no manager
run_query("""
    SELECT 
        e.emp_id,
        e.first_name || ' ' || e.last_name AS employee_name,
        m.first_name || ' ' || m.last_name AS manager_name
    FROM company.employees e
    LEFT JOIN company.employees m 
        ON e.manager_id = m.emp_id
    ORDER BY e.emp_id
""")

**Expected output:** ALL employees now appear. The directors (Alice Chen, Iris Taylor, Nathan Harris, etc.) will have `NULL` in the manager_name column because their `manager_id` is NULL and no match was found.

**Key difference:** INNER JOIN self-join lost the directors. LEFT JOIN self-join kept them with NULL managers. Choose based on whether you want to include top-level employees.

## Multi-Table Join — employees → departments → sales → products

Real analytical queries often join 3 or more tables. The golden rule: **build step by step**, verifying each join before adding the next.

**The chain:**
1. `employees` → `departments` (via department_id = dept_id)
2. + `sales` (via emp_id = employee_id)
3. + `products` (via product_id = product_id)

**Business question:** "Which employees sold which products, in which departments, and how much revenue did they generate?"

### Step 1: employees + departments

In [ ]:
# Step 1: Join employees with their departments
run_query("""
    SELECT 
        e.emp_id,
        e.first_name || ' ' || e.last_name AS employee_name,
        d.dept_name
    FROM company.employees e
    LEFT JOIN company.departments d 
        ON e.department_id = d.dept_id
    ORDER BY e.emp_id
""")

**Expected output:** All employees with their department names. This is the foundation — every subsequent join will build on this.

### Step 2: + sales (now we have employees + departments + sales)

In [ ]:
# Step 2: Add sales data
run_query("""
    SELECT 
        e.emp_id,
        e.first_name || ' ' || e.last_name AS employee_name,
        d.dept_name,
        s.sale_id,
        s.quantity,
        s.sale_date
    FROM company.employees e
    LEFT JOIN company.departments d 
        ON e.department_id = d.dept_id
    LEFT JOIN company.sales s 
        ON e.emp_id = s.employee_id
    ORDER BY e.emp_id, s.sale_date
    LIMIT 30
""")

**Expected output:** Employees with their department, and one row per sale. If an employee made multiple sales, their name appears multiple times (one row per sale). Employees with no sales will have NULL in sale_id, quantity, and sale_date.

**Notice the LEFT JOIN chain:** We use LEFT JOIN for both joins so we don't lose employees who haven't made sales yet. If we used INNER JOIN on sales, only employees with at least one sale would appear.

### Step 3: + products (the complete 4-table join)

In [ ]:
# Step 3: Add products for the full picture
run_query("""
    SELECT 
        e.emp_id,
        e.first_name || ' ' || e.last_name AS employee_name,
        d.dept_name,
        p.product_name,
        p.category,
        s.quantity,
        p.unit_price,
        s.quantity * p.unit_price AS revenue,
        s.sale_date
    FROM company.employees e
    LEFT JOIN company.departments d 
        ON e.department_id = d.dept_id
    LEFT JOIN company.sales s 
        ON e.emp_id = s.employee_id
    LEFT JOIN company.products p 
        ON s.product_id = p.product_id
    ORDER BY revenue DESC NULLS LAST
    LIMIT 30
""")

**Expected output:** A rich analytical result showing employee, department, product sold, quantity, price, calculated revenue, and date. The `NULLS LAST` in ORDER BY ensures employees with no sales appear at the bottom.

**Revenue calculation:** `s.quantity * p.unit_price AS revenue` — this computes revenue per sale line item directly in SQL. No need for Python/pandas post-processing!

**Building multi-table joins tip:** Always verify each step. If Step 2 produces unexpected results, adding Step 3 will only compound the problem.

In [ ]:
# Bonus: Revenue summary by department
run_query("""
    SELECT 
        d.dept_name,
        COUNT(s.sale_id) AS total_sales,
        SUM(s.quantity * p.unit_price) AS total_revenue,
        ROUND(AVG(s.quantity * p.unit_price), 2) AS avg_sale_value
    FROM company.departments d
    LEFT JOIN company.employees e 
        ON e.department_id = d.dept_id
    LEFT JOIN company.sales s 
        ON e.emp_id = s.employee_id
    LEFT JOIN company.products p 
        ON s.product_id = p.product_id
    GROUP BY d.dept_name
    ORDER BY total_revenue DESC NULLS LAST
""")

**Expected output:** A summary table with one row per department showing total number of sales, total revenue, and average sale value. Departments with no sales show NULL. This combines everything you've learned: multi-table joins, aggregation, and sorting.

**GROUP BY reminder:** When you aggregate (COUNT, SUM, AVG), every non-aggregated column in SELECT must appear in GROUP BY.

## Try It Yourself Exercises

Now it's your turn! Try each exercise before revealing the solution. Use what you've learned about different JOIN types.

### Exercise 1: Find employees who have never made a sale

Use a `LEFT JOIN` between `employees` and `sales`, then filter to find employees with no sales records. Show their name, department, and hire_date.

**Hint:** LEFT JOIN employees to sales, then WHERE the sale column IS NULL.

<details>
<summary><b>Click to reveal solution</b></summary>

```sql
SELECT 
    e.first_name || ' ' || e.last_name AS employee_name,
    d.dept_name,
    e.hire_date
FROM company.employees e
LEFT JOIN company.departments d 
    ON e.department_id = d.dept_id
LEFT JOIN company.sales s 
    ON e.emp_id = s.employee_id
WHERE s.sale_id IS NULL
ORDER BY e.last_name;
```

**Expected output:** All employees who have zero sales records. They will have their name, department, and hire_date, but no sales. This is useful for identifying employees who might need sales training or who are in non-sales roles.
</details>

In [ ]:
# Your solution here:
run_query("""
    
""")

### Exercise 2: Show each product and who sold it

Join `products` to `sales` to `employees` to see which employee sold which product. Include the product name, employee name, quantity, and sale date. Order by most recent sales first.

**Hint:** Start FROM products, LEFT JOIN sales, then LEFT JOIN employees. Use LEFT JOIN so products that have never been sold still appear.

<details>
<summary><b>Click to reveal solution</b></summary>

```sql
SELECT 
    p.product_name,
    p.category,
    e.first_name || ' ' || e.last_name AS sold_by,
    s.quantity,
    s.sale_date
FROM company.products p
LEFT JOIN company.sales s 
    ON p.product_id = s.product_id
LEFT JOIN company.employees e 
    ON s.employee_id = e.emp_id
ORDER BY s.sale_date DESC NULLS LAST;
```

**Expected output:** All products appear, even those never sold. For sold products, you'll see the employee name, quantity, and date. Products that were never sold have NULL in the employee, quantity, and date columns.
</details>

In [ ]:
# Your solution here:
run_query("""
    
""")

### Exercise 3: Find departments with no employees

Use a `LEFT JOIN` (or `RIGHT JOIN`) to find departments that currently have no employees assigned to them.

**Hint:** Start FROM departments, LEFT JOIN employees on department_id = dept_id, then WHERE emp_id IS NULL.

<details>
<summary><b>Click to reveal solution</b></summary>

```sql
SELECT 
    d.dept_name,
    d.location
FROM company.departments d
LEFT JOIN company.employees e 
    ON e.department_id = d.dept_id
WHERE e.emp_id IS NULL;
```

**Expected output:** Any departments that have zero employees assigned. This might be empty (if all departments have people) or might list one or two departments. This is a useful organizational report — "empty departments" that may need attention.
</details>

In [ ]:
# Your solution here:
run_query("""
    
""")